# 4 · Die Kampagne: Wen schreiben wir an?
**Päckli & Co.** · Data Science für Marketing · FHNW CAS Data Driven Marketing

Päckli & Co. hat die Kampagne an 9'000 zufällig gewählten Kunden getestet (Pilot): je 3'000 ohne E-Mail (`control`),
mit E-Mail (`email`) und mit E-Mail und 10 % Rabatt (`email_discount`). Jetzt entscheidet ihr für die übrigen
11'000 Kunden: **Wen schreiben wir an, und mit welchem Angebot?**

**Zahlen:** Marge 40 %, Rabatt 10 %, E-Mail CHF 0.10. Wer sich abmeldet, ist für künftige E-Mails verloren.
**Preis:** höchster zusätzlicher Gewinn der Kampagne.

**Churn-Risiko:** Damit alle mit denselben Risikogruppen arbeiten, rechnet diese Übung mit dem Modell der Musterlösung.
Euer eigenes Modell zählt im Churn-Vergleich.

**Pflicht:** Schritte 3 bis 5: ausführen, lesen, entscheiden, abgeben. Code schreibt ihr hier nur in der optionalen
Erweiterung E1. Schritte 1 und 2 zeigt die Kursleitung live: Ihr könnt sie einfach mit ausführen.

**Colab meldet «Zu viele Sitzungen»?** In den Notebooks der früheren Übungen: Laufzeit › «Verbindung trennen und Laufzeit löschen».

In [ ]:
#@title ▶ Setup: tragt eure Teamnummer ein und führt diese Zelle zuerst aus
TEAM = 0  #@param {type:"integer"}
BASIS_URL = "https://raw.githubusercontent.com/badrustanicki/fhnw-ds-marketing-2026/main/"
import os, sys, urllib.request
try:
    urllib.request.urlretrieve(BASIS_URL + "kurs.py", "kurs.py")   # immer die neuste Version
except OSError:
    if not os.path.exists("kurs.py"): raise
sys.modules.pop("kurs", None)   # neu laden, falls schon geladen
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from kurs import *
start(TEAM, BASIS_URL)

In [ ]:
#@title ▶ Vorbereitet: Churn-Risiko und Pilot-Ergebnis laden
churn_scores = catch_up("churn_scores", eigene=False)
pilot = catch_up("pilot_ergebnis", eigene=False).merge(churn_scores[["customer_id", "churn_risiko"]], on="customer_id")
pilot["risiko_gruppe"] = pd.cut(pilot["churn_risiko"], [0, 0.5, 0.7, 0.9, 1.0], include_lowest=True,
                                labels=["1 unter 50 %", "2 50 bis 70 %", "3 70 bis 90 %", "4 über 90 %"])
pilot.head()

## Teil A · Live mit der Kursleitung

### ▶ 1 Vorbereitet: Lohnt sich der Rabatt? Break-even
Marge 40 %, Rabatt 10 % auf den Listenpreis: Der Gewinn pro Bestellung sinkt von 40 % auf 30 % des Listenpreises. Um welchen Faktor muss die Kaufwahrscheinlichkeit mit Rabatt mindestens steigen, damit der erwartete Gewinn nicht sinkt? Gerechnet für eine treue Kundin (kauft sowieso mit 60 %) und einen gefährdeten Kunden (5 %).

In [ ]:
marge, rabatt = 0.40, 0.10
faktor = marge / (marge - rabatt)
print(f"Die Kaufwahrscheinlichkeit muss um den Faktor {faktor:.2f} steigen (+{faktor - 1:.0%}).")
for name, p in [("Treue Kundin", 0.60), ("Gefährdeter Kunde", 0.05)]:
    print(f"{name}: von {p:.0%} auf mindestens {p * faktor:.1%}")

**Lesen:** Treue Kundin: 80 %, kaum möglich. Gefährdeter Kunde: 6.7 %, machbar. Rabatt lohnt sich nur bei Kunden, die ohne ihn kaum kaufen.

### ▶ 2 Vorbereitet: Der Pilot: Was hat er gebracht?
Pro Gruppe: Anzahl Kunden, Kaufrate und Ø Gewinn pro Kunde. `profit` = Umsatz × 40 % − Rabatt − CHF 0.10 − erwarteter Abmelde-Verlust.

In [ ]:
pilot.groupby("arm").agg(kunden=("customer_id", "size"), kaufrate=("purchased", "mean"),
                         gewinn_pro_kunde=("profit", "mean")).round(2)

**Lesen:** Der Rabatt hat die höchste Kaufrate. Hat er auch den höchsten Gewinn? Es zählt der Gewinn **gegenüber der Kontrollgruppe**: Viele hätten auch ohne E-Mail gekauft.

## Teil B · Eure Kampagne

### ▶ 3 Vorbereitet: Bei wem wirkt die Kampagne?
Einfach ausführen und lesen. Der Pilot, aufgeteilt nach Churn-Risiko: Ø Gewinn pro Kunde mit E-Mail minus Ø Gewinn der Kontrollgruppe. Wo bringt die E-Mail Gewinn, wo kostet sie?

In [ ]:
t = pilot.pivot_table(index="risiko_gruppe", columns="arm", values="profit", aggfunc="mean", observed=True)
effekt = t[["email", "email_discount"]].sub(t["control"], axis=0).round(2)
effekt["pilotkunden"] = pilot.groupby("risiko_gruppe", observed=True).size()
effekt[["email", "email_discount"]].plot.bar(figsize=(7, 3.5), rot=0, title="Zusätzlicher Gewinn pro Kunde gegenüber der Kontrollgruppe (CHF)")
plt.axhline(0, color="black"); plt.xlabel("Churn-Risiko"); plt.show()
effekt

**Lesen:** **Balken über 0 = anschreiben, Balken unter 0 = weglassen.** Sichere Käufer (tiefes Risiko) kaufen sowieso: E-Mail und Rabatt kosten dort nur. Verlorene (sehr hohes Risiko) lesen keine E-Mail. Wo sind die Überzeugbaren? Die Gruppen sind verschieden gross (Spalte pilotkunden): Welches Angebot bringt über eure ganze Zielgruppe mehr?

### 4 Eure Entscheidung
Stellt die Regler auf die Gruppen aus Schritt 3, deren Balken über 0 liegen, und vergleicht beide Angebote. Die Hochrechnung stammt aus dem Pilot: Sie ist eine Schätzung, keine Garantie. Vorsicht beim Feintuning: Wer lange an den Reglern dreht, findet die Einstellung, die im Pilot zufällig am besten aussah.

In [ ]:
#@title 🎚️ Eure Entscheidung: Regler einstellen, dann ▶
min_risiko = 0.0  #@param {type:"slider", min:0, max:0.95, step:0.05}
max_risiko = 1.0  #@param {type:"slider", min:0.05, max:1, step:0.05}
angebot = "email"  #@param ["email", "email_discount"]
zielgruppe = churn_scores[churn_scores["churn_risiko"].between(min_risiko, max_risiko)
                          & ~churn_scores["customer_id"].isin(pilot["customer_id"])]
p = pilot[pilot["churn_risiko"].between(min_risiko, max_risiko)]
chf = lambda x: f"{x:+,.0f}".replace(",", "'")
zahl = lambda x: f"{x:,.0f}".replace(",", "'")
print(f"Zielgruppe: {zahl(len(zielgruppe))} Kunden mit Churn-Risiko {min_risiko:.0%} bis {max_risiko:.0%}, Angebot: {angebot}")
if min_risiko >= max_risiko:
    print("⚠️ «von» (min_risiko) muss kleiner sein als «bis» (max_risiko). Regler anpassen.")
elif len(zielgruppe) and (p["arm"] == angebot).any() and (p["arm"] == "control").any():
    pro_kunde = p.loc[p["arm"] == angebot, "profit"].mean() - p.loc[p["arm"] == "control", "profit"].mean()
    print(f"Im Pilot brachte das in dieser Gruppe {pro_kunde:+.2f} CHF pro Kunde.")
    print(f"Hochgerechnet auf die {zahl(0.8 * len(zielgruppe))} Kunden, die die E-Mail bekommen: {chf(pro_kunde * 0.8 * len(zielgruppe))} CHF")
    print("(20 % der Zielgruppe bleiben Kontrollgruppe.)")
    n_a, n_c = (p["arm"] == angebot).sum(), (p["arm"] == "control").sum()
    print(f"Grundlage: {zahl(n_a)} Pilotkunden mit {angebot}, {zahl(n_c)} ohne E-Mail.")
    if min(n_a, n_c) < 300:
        print("⚠️ Wenige Pilotkunden: Die Schätzung schwankt stark. Ein breiterer Bereich ist verlässlicher.")
else:
    print("⚠️ In diesem Bereich gibt es keine Kunden. Regler anpassen.")
ax = churn_scores["churn_risiko"].plot.hist(bins=40, color="lightgray", figsize=(7, 2.5), title="Alle Kunden nach Churn-Risiko")
ax.axvspan(min_risiko, max(min_risiko, max_risiko), color="gold", alpha=0.4); ax.set_xlabel("Churn-Risiko (gelb: eure Zielgruppe)"); plt.show()

### ▶ 5 Abgabe (Pflicht)
Eine Person pro Team reicht. Abgeben dürft ihr mehrmals, es zählt die letzte Abgabe. Colab fragt nach Google Drive: euer Konto wählen und alles zulassen. Klappt es nicht: die drei Werte in den Kurs-Chat schreiben.

In [ ]:
if "zielgruppe" not in globals() or len(zielgruppe) == 0:
    print("⚠️ Nicht gespeichert: Zuerst Schritt 4 ausführen und einen Bereich wählen, in dem es Kunden gibt.")
else:
    save(pd.DataFrame([{"min_risiko": min_risiko, "max_risiko": max_risiko, "angebot": angebot}]), "kampagne")
    print(f"Eure Kampagne: Churn-Risiko {min_risiko:.0%} bis {max_risiko:.0%}, Angebot {angebot}")

## Erweiterung (optional)

### E1 E-Mails schreiben lassen (optional)
**Ziel:** Personalisierung im grossen Stil, und wo sie schiefgeht. Wird nicht bewertet.

**Aufgabe:** Lade die Empfehlungen mit catch_up("empfehlungen", eigene=False) und die Produktnamen mit pd.read_parquet(BASIS_URL + "daten/products.parquet"). Zeige für 3 Kunden aus zielgruppe die Produktnamen (description) ihrer Empfehlungen. Schreibe dann für jeden eine kurze, persönliche E-Mail von Päckli & Co. mit diesen drei Produkten.

In [ ]:
# Euer Code hier, dann ▶

**Plausibel?** Würdet ihr die E-Mail öffnen? Erfindet die KI Rabatte oder Produkte?